# **Agent 1 with Stable Baseline Reward Function**

In [ ]:
!pip install stable-baselines3[extra] gymnasium[mujoco] swig

In [ ]:
import gymnasium as gym
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
from stable_baselines3.common.monitor import Monitor
print("✅ All imports successful!")

In [ ]:
# Create evaluation environment and callbacks

train_env = make_vec_env("HalfCheetah-v4", n_envs=1)

eval_env = gym.make("HalfCheetah-v4")

# Stop training when reward reaches 3000
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=3000, verbose=1)

# Auto-evaluate and save best model
eval_callback = EvalCallback(
    eval_env,
    callback_on_new_best=stop_callback,
    eval_freq=10000,
    best_model_save_path="./best_model/",
    verbose=1
)

print("✅ eval_callback is now defined!")

In [ ]:
# Create the SAC model with built-in MLP policy
model = SAC(
    "MlpPolicy",           # Neural network with 2 hidden layers (default: 256, 256)
    train_env,
    verbose=1,             # Shows training updates
    learning_rate=0.0003,  # Default SAC learning rate
    buffer_size=1000000,   # Replay buffer size
    batch_size=256,        # Batch size for training
    tau=0.005,             # Soft update coefficient
    gamma=0.99,            # Discount factor
    train_freq=1,          # Update model every step
    gradient_steps=1,      # Do 1 gradient step per environment step
    device="auto"          # Automatically uses GPU if available
)

print("✅ SAC Agent Created!")
print(f"   Policy: MlpPolicy (2x256 hidden layers)")
print(f"   Device: {model.device}")

In [ ]:
# Train the model
print("🚀 Starting training...")
print("⏱️ With GPU, this will take ~10-15 minutes")
print("📊 Training progress below:\n")

model.learn(
    total_timesteps=200_000,
    callback=eval_callback,
    progress_bar=True
)

print("\n✅ Training complete!")

In [ ]:
import imageio
import numpy as np
from stable_baselines3 import SAC
import gymnasium as gym
from IPython.display import Image, display
import time

print("🎬 Recording cheetah (manual frame capture)...")

best_model = SAC.load("./best_model/best_model")

# Try different render modes
render_modes = ["rgb_array", "rgb_array_list", "rgb_array"]

frames = []
total_reward = 0

for mode in render_modes:
    try:
        print(f"Trying render_mode='{mode}'...")
        env = gym.make("HalfCheetah-v4", render_mode=mode)
        obs, _ = env.reset()

        for step in range(200):  # Capture 200 frames
            # Get frame
            try:
                frame = env.render()
                if frame is not None and isinstance(frame, np.ndarray):
                    frames.append(frame)
            except:
                pass

            # Take action
            action, _ = best_model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward

            if terminated or truncated:
                break

        env.close()

        if len(frames) > 0:
            print(f"✅ Successfully captured {len(frames)} frames with mode='{mode}'")
            break

    except Exception as e:
        print(f"Mode '{mode}' failed: {e}")
        continue

if frames:
    # Create GIF
    os.makedirs("videos", exist_ok=True)
    gif_path = "videos/halfcheetah_trained.gif"

    # Resize frames if needed (optional - makes GIF smaller)
    resized_frames = []
    for frame in frames[::2]:  # Take every 2nd frame
        if frame.shape[0] > 400:
            from PIL import Image as PILImage
            img = PILImage.fromarray(frame)
            img = img.resize((img.width//2, img.height//2))
            resized_frames.append(np.array(img))
        else:
            resized_frames.append(frame)

    imageio.mimsave(gif_path, resized_frames, fps=30, loop=0)
    print(f"✅ GIF saved!")
    display(Image(filename=gif_path, width=600))
else:
    print("Could not capture frames in Colab.")
    print("\n💡 Last resort: Download model and run locally:")
    print("   from google.colab import files")
    print("   files.download('./best_model/best_model.zip')")


In [ ]:
from google.colab import files

# Download your trained model
files.download("./best_model/best_model.zip")
print("✅ Model downloaded!")

# Save this script to run on your computer
script = '''
import gymnasium as gym
from stable_baselines3 import SAC
import imageio
import numpy as np

# Load model
model = SAC.load("best_model.zip")

# Create environment with display
env = gym.make("HalfCheetah-v4", render_mode="rgb_array")

frames = []
obs, _ = env.reset()

for step in range(1000):
    frame = env.render()
    frames.append(frame)
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break

env.close()

# Save as GIF
imageio.mimsave("halfcheetah.gif", frames[::2], fps=30)
print("GIF saved!")
'''

with open("visualize_cheetah.py", "w") as f:
    f.write(script)

files.download("visualize_cheetah.py")
print("✅ Visualization script downloaded!")

print("\n📝 On your computer, run:")
print("   pip install gymnasium[mujoco] stable-baselines3 imageio")
print("   python visualize_cheetah.py")


# **Agent 2 with Custom Reward Shaping**

In [ ]:
# Cell 1: Environment Setup and Analysis
!pip install gymnasium[mujoco] mujoco pygame

import gymnasium as gym
import numpy as np

# Create environment
env = gym.make("HalfCheetah-v4")

# Print environment specs
print("="*50)
print("ENVIRONMENT ANALYSIS")
print("="*50)
print(f"Observation Space: {env.observation_space}")
print(f"  - Shape: {env.observation_space.shape}")
print(f"  - Type: {env.observation_space.dtype}")
print(f"\nAction Space: {env.action_space}")
print(f"  - Shape: {env.action_space.shape}")
print(f"  - Range: [{env.action_space.low[0]:.2f}, {env.action_space.high[0]:.2f}]")
print(f"  - Type: {env.action_space.dtype}")

# Test random actions to understand reward structure
print("\n" + "="*50)
print("REWARD STRUCTURE ANALYSIS (100 random steps)")
print("="*50)

rewards = []
forward_velocities = []
control_costs = []

state, _ = env.reset()
for step in range(100):
    action = env.action_space.sample()
    next_state, reward, terminated, truncated, info = env.step(action)

    # The default HalfCheetah reward = forward_velocity - control_cost
    # Let's extract what's happening
    if 'reward_run' in info:
        forward_velocities.append(info['reward_run'])
    if 'reward_ctrl' in info:
        control_costs.append(info['reward_ctrl'])

    rewards.append(reward)

    if terminated or truncated:
        state, _ = env.reset()
    else:
        state = next_state

print(f"Mean reward per step: {np.mean(rewards):.4f}")
print(f"Std reward per step: {np.std(rewards):.4f}")
print(f"Min reward: {np.min(rewards):.4f}")
print(f"Max reward: {np.max(rewards):.4f}")
print(f"\nReward breakdown (if available):")
print(f"  Forward velocity component mean: {np.mean(forward_velocities) if forward_velocities else 'N/A'}")
print(f"  Control cost component mean: {np.mean(control_costs) if control_costs else 'N/A'}")

# Test a simple heuristic: just moving forward vs staying still
print("\n" + "="*50)
print("HEURISTIC TEST: Apply forward force")
print("="*50)

# Try pushing all joints forward slightly
forward_action = np.array([0.5] * env.action_space.shape[0])
state, _ = env.reset()
total_reward = 0
for step in range(50):
    state, reward, terminated, truncated, _ = env.step(forward_action)
    total_reward += reward
print(f"Total reward over 50 steps with forward bias: {total_reward:.2f}")

# Try zero action (do nothing)
zero_action = np.zeros(env.action_space.shape[0])
state, _ = env.reset()
total_reward_zero = 0
for step in range(50):
    state, reward, terminated, truncated, _ = env.step(zero_action)
    total_reward_zero += reward
print(f"Total reward over 50 steps with zero action: {total_reward_zero:.2f}")

env.close()

print("\n Cell 1 complete. Based on these results, what do you observe?")
print("   The cheetah gets reward for moving forward, but pays a cost for large actions.")
print("   This tells us we need to balance exploration vs efficiency.")

In [ ]:
# Cell 2: Custom Reward Shaping Wrapper
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class ShapedHalfCheetahEnv(gym.Wrapper):
    """
    Reward shaping wrapper for HalfCheetah.
    Original reward: forward_velocity - control_cost
    Our modifications:
    1. Scale reward by 10x (to get gradients in reasonable range [-10, +30])
    2. Add small survival bonus (+0.1 per step) to counteract control cost penalty
    3. Clip extreme values to prevent destabilization
    """
    def __init__(self, env, reward_scale=10.0, survival_bonus=0.05):
        super().__init__(env)
        self.reward_scale = reward_scale
        self.survival_bonus = survival_bonus
        self.episode_length = 0

    def step(self, action):
        next_state, original_reward, terminated, truncated, info = self.env.step(action)

        # Apply reward shaping
        shaped_reward = original_reward * self.reward_scale + self.survival_bonus

        # Clip to reasonable range to prevent exploding gradients
        shaped_reward = np.clip(shaped_reward, -20.0, 50.0)

        self.episode_length += 1

        # Log original and shaped rewards for monitoring
        info['original_reward'] = original_reward
        info['shaped_reward'] = shaped_reward

        return next_state, shaped_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.episode_length = 0
        return self.env.reset(**kwargs)

# Test the shaped environment
print("="*50)
print("TESTING REWARD SHAPING")
print("="*50)

# Create shaped environment
base_env = gym.make("HalfCheetah-v4")
shaped_env = ShapedHalfCheetahEnv(base_env, reward_scale=10.0, survival_bonus=0.05)

# Test forward bias action with shaping
forward_action = np.array([0.5] * shaped_env.action_space.shape[0])
state, _ = shaped_env.reset()
total_original = 0
total_shaped = 0

for step in range(50):
    state, shaped_reward, terminated, truncated, info = shaped_env.step(forward_action)
    total_shaped += shaped_reward
    total_original += info['original_reward']

print(f"Original total reward (50 steps forward bias): {total_original:.2f}")
print(f"Shaped total reward (50 steps forward bias): {total_shaped:.2f}")
print(f"Reward scale factor applied: ~{total_shaped/total_original:.1f}x")

# Test random actions with shaping
state, _ = shaped_env.reset()
random_rewards = []
for step in range(100):
    action = shaped_env.action_space.sample()
    _, reward, _, _, _ = shaped_env.step(action)
    random_rewards.append(reward)

print(f"\nShaped random action mean reward: {np.mean(random_rewards):.3f}")
print(f"Shaped random action std: {np.std(random_rewards):.3f}")

# Test zero action
state, _ = shaped_env.reset()
zero_action = np.zeros(shaped_env.action_space.shape[0])
zero_rewards = []
for step in range(50):
    _, reward, _, _, _ = shaped_env.step(zero_action)
    zero_rewards.append(reward)

print(f"\nShaped zero action mean reward: {np.mean(zero_rewards):.3f}")

shaped_env.close()
base_env.close()

print("\n" + "="*50)
print("REWARD STATISTICS AFTER SHAPING")
print("="*50)
print(f"Expected per-step reward range: [-20, +50]")
print(f"Random policy mean: {np.mean(random_rewards):.2f} (was -0.057)")
print(f"This gives reasonable gradient magnitudes for neural networks")

print("\n Cell 2 complete. The reward is now scaled appropriately.")
print("   Question: Should we increase survival_bonus to 0.1 or keep at 0.05?")
print("   (Higher bonus = more incentive to stay alive, but less efficient running)")

In [ ]:
# Cell 3: Aggressive Reward Restructuring
import gymnasium as gym
import numpy as np

class AggressiveShapedHalfCheetah(gym.Wrapper):
    """
    Restructured reward for HalfCheetah.
    Problem: Original reward discourages movement entirely.
    Solution:
    1. Extract forward velocity directly from simulation
    2. Reward forward velocity heavily (+1.0 per m/s)
    3. Small penalty for control (-0.05 per action magnitude)
    4. Bonus for exceeding speed threshold
    """
    def __init__(self, env, forward_reward_scale=1.0, control_penalty=0.05, speed_bonus_threshold=2.0):
        super().__init__(env)
        self.forward_reward_scale = forward_reward_scale
        self.control_penalty = control_penalty
        self.speed_bonus_threshold = speed_bonus_threshold
        self.episode_length = 0

    def step(self, action):
        next_state, original_reward, terminated, truncated, info = self.env.step(action)

        # Extract forward velocity (index 8 in observation is root x-velocity)
        # But let's get it from info which is more reliable
        forward_vel = info.get('reward_run', 0.0)

        # If info doesn't have it, estimate from state difference (less accurate)
        if forward_vel == 0.0:
            # Alternative: use x-position from observation? Better to use what we have
            forward_vel = original_reward  # Fallback, not ideal

        # Calculate custom reward
        forward_reward = forward_vel * self.forward_reward_scale

        # Control penalty (encourage efficient movements)
        control_cost = np.square(action).mean() * self.control_penalty

        # Speed bonus (encourage actually running)
        speed_bonus = 0.5 if forward_vel > self.speed_bonus_threshold else 0.0

        # Survival bonus (small positive for staying alive)
        survival_bonus = 0.02

        # Combine rewards
        shaped_reward = forward_reward - control_cost + survival_bonus + speed_bonus

        # Clip to reasonable range
        shaped_reward = np.clip(shaped_reward, -5.0, 20.0)

        self.episode_length += 1

        # Store debug info
        info['forward_vel'] = forward_vel
        info['forward_reward'] = forward_reward
        info['control_cost'] = control_cost
        info['shaped_reward'] = shaped_reward
        info['original_reward'] = original_reward

        return next_state, shaped_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.episode_length = 0
        return self.env.reset(**kwargs)

# Test the new aggressive shaping
print("="*50)
print("TESTING AGGRESSIVE REWARD RESTRUCTURING")
print("="*50)

base_env = gym.make("HalfCheetah-v4")
aggressive_env = AggressiveShapedHalfCheetah(base_env, forward_reward_scale=1.0, control_penalty=0.05)

# Test forward bias action
forward_action = np.array([0.5] * aggressive_env.action_space.shape[0])
state, _ = aggressive_env.reset()
forward_rewards = []
forward_vels = []

for step in range(50):
    state, reward, terminated, truncated, info = aggressive_env.step(forward_action)
    forward_rewards.append(reward)
    forward_vels.append(info['forward_vel'])

print(f"Forward bias action results (50 steps):")
print(f"  Mean shaped reward: {np.mean(forward_rewards):.3f}")
print(f"  Total shaped reward: {np.sum(forward_rewards):.2f}")
print(f"  Mean forward velocity: {np.mean(forward_vels):.3f} m/s")

# Test zero action
zero_action = np.zeros(aggressive_env.action_space.shape[0])
state, _ = aggressive_env.reset()
zero_rewards = []

for step in range(50):
    state, reward, _, _, info = aggressive_env.step(zero_action)
    zero_rewards.append(reward)

print(f"\nZero action results (50 steps):")
print(f"  Mean shaped reward: {np.mean(zero_rewards):.3f}")
print(f"  Total shaped reward: {np.sum(zero_rewards):.2f}")

# Test with increasing action scales
print(f"\nTesting different action intensities:")
for scale in [0.2, 0.5, 0.8, 1.0]:
    test_action = np.array([scale] * aggressive_env.action_space.shape[0])
    state, _ = aggressive_env.reset()
    total_reward = 0
    for step in range(50):
        state, reward, _, _, _ = aggressive_env.step(test_action)
        total_reward += reward
    print(f"  Scale {scale}: Total reward = {total_reward:.2f}")

# Find best action magnitude with random search
print(f"\nFinding optimal action magnitude (random sampling):")
best_reward = -float('inf')
best_scale = 0
for _ in range(20):
    scale = np.random.uniform(0.1, 1.2)
    test_action = np.array([scale] * aggressive_env.action_space.shape[0])
    state, _ = aggressive_env.reset()
    total_reward = 0
    for step in range(50):
        state, reward, _, _, _ = aggressive_env.step(test_action)
        total_reward += reward
    if total_reward > best_reward:
        best_reward = total_reward
        best_scale = scale
    print(f"  Scale {scale:.2f}: Total reward = {total_reward:.2f}")

print(f"\n✅ Best action scale found: {best_scale:.2f} with reward {best_reward:.2f}")

aggressive_env.close()
base_env.close()

print("\n" + "="*50)
print("CRITICAL OBSERVATIONS:")
print("="*50)
if np.mean(forward_rewards) > 0:
    print("✅ GOOD: Forward bias now gives positive reward!")
    print(f"   The cheetah will learn to move forward.")
else:
    print("❌ WARNING: Forward bias still gives negative reward.")
    print("   Need to increase forward_reward_scale or decrease control_penalty.")

print(f"\nZero action reward: {np.mean(zero_rewards):.3f}")
if np.mean(zero_rewards) < np.mean(forward_rewards):
    print("✅ GOOD: Moving forward is better than standing still.")
else:
    print("❌ WARNING: Standing still is still better than moving.")

print("\nNext step depends on these results. Share the output and we'll:")
print("  - Adjust forward_reward_scale if needed (try 2.0 or 3.0)")
print("  - Or proceed to network architecture if reward is positive")

In [ ]:
# Cell 4: Optimized SAC Implementation for HalfCheetah
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


# 1. Orthogonal Initialization
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
        nn.init.constant_(m.bias, 0.0)

# 2. Separate Learning Rates
self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=1e-4)
self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=3e-4)

# 3. Gradient Clipping
torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)

# 4. Clamped Log Std
log_std = torch.clamp(log_std, -10, 2)



# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters tuned based on our reward analysis
class Config:
    # Environment
    env_name = "HalfCheetah-v4"

    # Network architecture
    hidden_dim = 256
    num_layers = 2

    # Optimizer
    actor_lr = 1e-4  # Smaller because our reward scale is moderate
    critic_lr = 3e-4  # Critics can learn faster

    # SAC specific
    gamma = 0.99  # Discount factor
    tau = 0.005   # Soft update target
    alpha = 0.1   # Initial temperature (smaller = less exploration)
    target_entropy = -6.0  # -dim(action) = -6

    # Training
    buffer_size = int(1e6)
    batch_size = 256
    start_steps = 5000  # Less random exploration needed
    update_every = 1
    train_after = 5000

    # Reward shaping parameters (from our tests)
    forward_reward_scale = 1.0
    control_penalty = 0.05
    survival_bonus = 0.02

    # Checkpoints
    save_frequency = 5000
    eval_frequency = 5000

config = Config()

# Custom layer initialization for better training stability
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
        nn.init.constant_(m.bias, 0.0)

# Actor Network (Policy) with Tanh Normal distribution
class TanhNormalActor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action, hidden_dim=256):
        super().__init__()
        self.max_action = max_action

        # Shared backbone
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # Mean and log std heads
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)

        self.apply(init_weights)

    def forward(self, state):
        features = self.shared(state)
        mean = self.mean(features)
        log_std = self.log_std(features)

        # Constrain log_std for stable training
        log_std = torch.clamp(log_std, -10, 2)
        std = log_std.exp()

        return mean, std

    def sample(self, state, deterministic=False):
        mean, std = self.forward(state)

        if deterministic:
            action = torch.tanh(mean)
            log_prob = None
        else:
            normal = torch.distributions.Normal(mean, std)
            z = normal.rsample()  # Reparameterization trick
            action = torch.tanh(z)

            # Compute log probability with tanh correction
            log_prob = normal.log_prob(z) - torch.log(1 - action.pow(2) + 1e-6)
            log_prob = log_prob.sum(-1, keepdim=True)

        return action * self.max_action, log_prob

# Critic Network (Ensemble of 2 Q-functions)
class DoubleCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()

        # Q1 network
        self.q1 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        # Q2 network
        self.q2 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        self.apply(init_weights)

    def forward(self, state, action):
        sa = torch.cat([state, action], dim=-1)
        q1 = self.q1(sa)
        q2 = self.q2(sa)
        return q1, q2

# Test network shapes
print("="*50)
print("NETWORK ARCHITECTURE VALIDATION")
print("="*50)

# Create dummy environment to get dimensions
temp_env = gym.make("HalfCheetah-v4")
state_dim = temp_env.observation_space.shape[0]
action_dim = temp_env.action_space.shape[0]
max_action = float(temp_env.action_space.high[0])

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Max action: {max_action}")

# Test actor
actor = TanhNormalActor(state_dim, action_dim, max_action, config.hidden_dim)
test_state = torch.randn(4, state_dim)
test_action, test_log_prob = actor.sample(test_state)
print(f"\nActor output shape: {test_action.shape}")
print(f"Log prob shape: {test_log_prob.shape if test_log_prob is not None else 'None'}")
print(f"Action range: [{test_action.min().item():.2f}, {test_action.max().item():.2f}]")

# Test critic
critic = DoubleCritic(state_dim, action_dim, config.hidden_dim)
test_q1, test_q2 = critic(test_state, test_action)
print(f"\nCritic output shapes: Q1={test_q1.shape}, Q2={test_q2.shape}")

# Count parameters
actor_params = sum(p.numel() for p in actor.parameters())
critic_params = sum(p.numel() for p in critic.parameters())
print(f"\nTotal parameters:")
print(f"  Actor: {actor_params:,}")
print(f"  Critic: {critic_params:,}")
print(f"  Total: {actor_params + critic_params:,}")

# Test forward pass through network
print("\nTesting forward pass...")
with torch.no_grad():
    for _ in range(10):
        state = torch.randn(config.batch_size, state_dim)
        action, log_prob = actor.sample(state)
        q1, q2 = critic(state, action)

        assert not torch.isnan(action).any(), "NaN in action"
        assert not torch.isnan(log_prob).any(), "NaN in log_prob"
        assert not torch.isnan(q1).any(), "NaN in Q1"
        assert not torch.isnan(q2).any(), "NaN in Q2"

print("✅ All network tests passed! No NaN values.")

temp_env.close()

print("\n" + "="*50)
print("HYPERPARAMETER SUMMARY")
print("="*50)
print(f"Learning rates - Actor: {config.actor_lr}, Critic: {config.critic_lr}")
print(f"Temperature (alpha): {config.alpha}")
print(f"Target entropy: {config.target_entropy}")
print(f"Batch size: {config.batch_size}")
print(f"Start steps: {config.start_steps}")
print("\nKey design decisions:")
print("1. Smaller actor LR (1e-4) prevents policy from changing too fast")
print("2. Lower alpha (0.1) reduces exploration since we have good reward shaping")
print("3. Orthogonal initialization for stable gradients")
print("4. Clamped log_std to prevent extreme exploration")

print("\n✅ Cell 4 complete. Architecture is ready for training.")
print("\nNext: Cell 5 will implement the Replay Buffer and training loop.")

In [ ]:
# Cell 5: Optimized Training Loop with Reward Shaping
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import deque
import random
import time
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Replay Buffer (optimized for performance)
class ReplayBuffer:
    def __init__(self, capacity, state_dim, action_dim, device):
        self.capacity = capacity
        self.device = device
        self.ptr = 0
        self.size = 0

        # Pre-allocate memory as numpy arrays (faster)
        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros((capacity, action_dim), dtype=np.float32)
        self.rewards = np.zeros((capacity, 1), dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones = np.zeros((capacity, 1), dtype=np.float32)

    def push(self, state, action, reward, next_state, done):
        # Store in numpy (faster than list of tuples)
        self.states[self.ptr] = state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.next_states[self.ptr] = next_state
        self.dones[self.ptr] = done

        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        # Random indices
        idxs = np.random.randint(0, self.size, size=batch_size)

        # Convert to torch tensors on device
        states = torch.FloatTensor(self.states[idxs]).to(self.device)
        actions = torch.FloatTensor(self.actions[idxs]).to(self.device)
        rewards = torch.FloatTensor(self.rewards[idxs]).to(self.device)
        next_states = torch.FloatTensor(self.next_states[idxs]).to(self.device)
        dones = torch.FloatTensor(self.dones[idxs]).to(self.device)

        return states, actions, rewards, next_states, dones

    def __len__(self):
        return self.size

# Reward Shaping Wrapper (from our earlier tests)
class AggressiveShapedHalfCheetah(gym.Wrapper):
    def __init__(self, env, forward_reward_scale=1.0, control_penalty=0.05, survival_bonus=0.02):
        super().__init__(env)
        self.forward_reward_scale = forward_reward_scale
        self.control_penalty = control_penalty
        self.survival_bonus = survival_bonus

    def step(self, action):
        next_state, original_reward, terminated, truncated, info = self.env.step(action)

        # Extract forward velocity safely
        forward_vel = info.get('reward_run', 0.0)

        # Custom reward calculation (proven effective in Cell 3)
        forward_reward = forward_vel * self.forward_reward_scale
        control_cost = np.square(action).mean() * self.control_penalty

        shaped_reward = forward_reward - control_cost + self.survival_bonus
        shaped_reward = np.clip(shaped_reward, -5.0, 20.0)

        info['shaped_reward'] = shaped_reward
        info['original_reward'] = original_reward
        info['forward_vel'] = forward_vel

        return next_state, shaped_reward, terminated, truncated, info

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

# SAC Agent (Complete implementation)
class SACAgent:
    def __init__(self, state_dim, action_dim, max_action, config, device):
        self.actor = TanhNormalActor(state_dim, action_dim, max_action, config.hidden_dim).to(device)
        self.critic = DoubleCritic(state_dim, action_dim, config.hidden_dim).to(device)
        self.target_critic = DoubleCritic(state_dim, action_dim, config.hidden_dim).to(device)

        # Copy target networks
        self.target_critic.load_state_dict(self.critic.state_dict())

        # Optimizers with different learning rates
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=config.actor_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=config.critic_lr)

        # Learnable temperature (alpha)
        self.target_entropy = -action_dim
        self.log_alpha = torch.tensor(np.log(config.alpha), requires_grad=True, device=device)
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=config.actor_lr)

        self.max_action = max_action
        self.config = config
        self.device = device

        # Metrics tracking
        self.actor_losses = []
        self.critic_losses = []
        self.alphas = []

    def select_action(self, state, evaluate=False):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)

        with torch.no_grad():
            if evaluate:
                # Deterministic for evaluation
                action, _ = self.actor.sample(state, deterministic=True)
            else:
                action, _ = self.actor.sample(state, deterministic=False)

        return action.cpu().numpy()[0]

    def train(self, replay_buffer, batch_size):
        if len(replay_buffer) < batch_size:
            return None, None, None

        # Sample batch
        states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

        # === Update Critic ===
        with torch.no_grad():
            # Sample next actions for target
            next_actions, next_log_probs = self.actor.sample(next_states)

            # Compute target Q values
            target_q1, target_q2 = self.target_critic(next_states, next_actions)
            target_q = torch.min(target_q1, target_q2)

            # Temperature for entropy
            alpha = self.log_alpha.exp()
            target_q = target_q - alpha * next_log_probs

            # Bellman backup
            target_q = rewards + (1 - dones) * self.config.gamma * target_q

        # Current Q values
        current_q1, current_q2 = self.critic(states, actions)
        critic_loss = nn.MSELoss()(current_q1, target_q) + nn.MSELoss()(current_q2, target_q)

        # Optimize critic
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)  # Gradient clipping
        self.critic_optimizer.step()

        # === Update Actor ===
        new_actions, log_probs = self.actor.sample(states)
        q1, q2 = self.critic(states, new_actions)
        q = torch.min(q1, q2)

        alpha = self.log_alpha.exp().detach()
        actor_loss = (alpha * log_probs - q).mean()

        # Optimize actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
        self.actor_optimizer.step()

        # === Update Temperature ===
        alpha_loss = -(self.log_alpha * (log_probs + self.target_entropy).detach()).mean()
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()

        # === Soft Update Target Networks ===
        with torch.no_grad():
            for param, target_param in zip(self.critic.parameters(), self.target_critic.parameters()):
                target_param.data.copy_(self.config.tau * param.data + (1 - self.config.tau) * target_param.data)

        # Store metrics
        current_alpha = alpha.item()
        self.actor_losses.append(actor_loss.item())
        self.critic_losses.append(critic_loss.item())
        self.alphas.append(current_alpha)

        return actor_loss.item(), critic_loss.item(), current_alpha

    def save(self, path):
        torch.save({
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict(),
            'target_critic': self.target_critic.state_dict(),
            'actor_optimizer': self.actor_optimizer.state_dict(),
            'critic_optimizer': self.critic_optimizer.state_dict(),
        }, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.actor.load_state_dict(checkpoint['actor'])
        self.critic.load_state_dict(checkpoint['critic'])
        self.target_critic.load_state_dict(checkpoint['target_critic'])

# Training function
def train():
    # Create environment with reward shaping
    base_env = gym.make(config.env_name)
    env = AggressiveShapedHalfCheetah(
        base_env,
        forward_reward_scale=config.forward_reward_scale,
        control_penalty=config.control_penalty,
        survival_bonus=config.survival_bonus
    )

    # Get dimensions
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    max_action = float(env.action_space.high[0])

    print(f"Training on {config.env_name}")
    print(f"State dim: {state_dim}, Action dim: {action_dim}")
    print(f"Max action: {max_action}")
    print(f"Device: {device}")
    print("-" * 50)

    # Initialize agent and replay buffer
    agent = SACAgent(state_dim, action_dim, max_action, config, device)
    replay_buffer = ReplayBuffer(config.buffer_size, state_dim, action_dim, device)

    # Training metrics
    episode_rewards = []
    episode_lengths = []
    episode_rewards_shaped = []
    best_reward = -float('inf')

    # Training loop
    state, _ = env.reset()
    episode_reward = 0
    episode_reward_shaped = 0
    episode_step = 0
    episode_num = 0
    total_steps = 0

    # Progress bar
    pbar = tqdm(total=config.total_timesteps, desc="Training")

    for step in range(1, config.total_timesteps + 1):
        # Select action (with exploration for initial steps)
        if total_steps < config.start_steps:
            action = env.action_space.sample()
        else:
            action = agent.select_action(state, evaluate=False)

        # Take step
        next_state, reward_shaped, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # Get original reward for logging
        reward_original = info.get('original_reward', reward_shaped)

        # Store transition
        replay_buffer.push(state, action, reward_shaped, next_state, float(done))

        # Update metrics
        episode_reward += reward_original
        episode_reward_shaped += reward_shaped
        episode_step += 1
        total_steps += 1
        state = next_state

        # Train agent
        if total_steps >= config.start_steps and total_steps % config.update_every == 0:
            agent.train(replay_buffer, config.batch_size)

        if done:
            # Episode finished
            episode_rewards.append(episode_reward)
            episode_rewards_shaped.append(episode_reward_shaped)
            episode_lengths.append(episode_step)

            # Checkpoint best model
            if episode_reward > best_reward:
                best_reward = episode_reward
                agent.save("best_cheetah.pth")
                print(f"\n✨ New best reward: {best_reward:.2f}")

            # Print progress
            if len(episode_rewards) % 10 == 0:
                avg_reward = np.mean(episode_rewards[-10:])
                avg_length = np.mean(episode_lengths[-10:])
                avg_shaped = np.mean(episode_rewards_shaped[-10:])
                print(f"\nEp: {episode_num} | Steps: {total_steps} | "
                      f"Avg Reward (10 eps): {avg_reward:.1f} | "
                      f"Avg Shaped: {avg_shaped:.1f} | "
                      f"Avg Length: {avg_length:.0f}")

            # Save periodic checkpoint
            if episode_num % 50 == 0 and episode_num > 0:
                agent.save(f"checkpoint_ep{episode_num}.pth")

            # Reset environment
            state, _ = env.reset()
            episode_reward = 0
            episode_reward_shaped = 0
            episode_step = 0
            episode_num += 1

        pbar.update(1)
        pbar.set_postfix({
            'Ep': episode_num,
            'Rew': episode_reward if episode_reward != 0 else 0,
            'Best': best_reward
        })

    pbar.close()
    env.close()
    base_env.close()

    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0,0].plot(episode_rewards)
    axes[0,0].set_title('Episode Rewards (Original)')
    axes[0,0].set_xlabel('Episode')
    axes[0,0].set_ylabel('Total Reward')
    axes[0,0].grid(True, alpha=0.3)

    axes[0,1].plot(episode_rewards_shaped)
    axes[0,1].set_title('Episode Rewards (Shaped)')
    axes[0,1].set_xlabel('Episode')
    axes[0,1].set_ylabel('Shaped Reward')
    axes[0,1].grid(True, alpha=0.3)

    # Smooth rewards for trend
    if len(episode_rewards) > 10:
        smoothed = np.convolve(episode_rewards, np.ones(10)/10, mode='valid')
        axes[1,0].plot(smoothed)
        axes[1,0].set_title('Smoothed Reward (10-episode MA)')
    else:
        axes[1,0].plot(episode_rewards)
        axes[1,0].set_title('Episode Rewards')
    axes[1,0].set_xlabel('Episode')
    axes[1,0].set_ylabel('Reward')
    axes[1,0].grid(True, alpha=0.3)

    axes[1,1].hist(episode_rewards, bins=30, alpha=0.7)
    axes[1,1].set_title('Reward Distribution')
    axes[1,1].set_xlabel('Total Reward')
    axes[1,1].set_ylabel('Frequency')
    axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150)
    plt.show()

    print("\n" + "="*50)
    print("TRAINING COMPLETE")
    print("="*50)
    print(f"Best reward: {best_reward:.2f}")
    print(f"Final avg reward (last 10 eps): {np.mean(episode_rewards[-10:]):.2f}")
    print(f"Total episodes: {episode_num}")
    print(f"Total timesteps: {total_steps}")
    print("\nModels saved: best_cheetah.pth")

    return agent, episode_rewards, episode_rewards_shaped

# Set total timesteps (adjust based on your patience)
config.total_timesteps = 100000  # 100k steps - about 10-15 minutes

print("="*50)
print("READY TO TRAIN")
print("="*50)
print(f"Total timesteps: {config.total_timesteps:,}")
print(f"Start steps (random): {config.start_steps:,}")
print(f"Expected training time: ~10-15 minutes")
print("\n⚠️ IMPORTANT: This will train for 100k steps.")
print("   You'll see progress updates every 10 episodes.")
print("   The cheetah should learn to run forward within 30-40 episodes.")
print("\n→ Run the next cell to start training!")

# Test that the environment works
test_env = gym.make(config.env_name)
test_shaped = AggressiveShapedHalfCheetah(test_env)
test_state, _ = test_shaped.reset()
test_action = test_shaped.action_space.sample()
test_next, test_reward, _, _, _ = test_shaped.step(test_action)
print(f"\n✅ Environment test passed!")
print(f"   Sample reward: {test_reward:.3f}")
test_shaped.close()
test_env.close()

In [ ]:
# Cell 6: Run Training
# This will train for 100,000 timesteps (about 10-15 minutes)

# Override any config settings if needed
config.total_timesteps = 100000  # Keep at 100k for initial test

# Start training
print("🚀 TRAINING STARTED")
print("="*50)
print(f"This will take {config.total_timesteps/1000:.0f}k steps")
print("Watch for the cheetah to start running around episode 20-30")
print("\nProgress updates every 10 episodes:")
print("-"*50)

# Run training
trained_agent, rewards, shaped_rewards = train()

print("\n" + "="*50)
print("🎉 TRAINING COMPLETE!")
print("="*50)
print(f"Final stats:")
print(f"  Best episode reward: {max(rewards):.2f}")
print(f"  Average last 10 episodes: {np.mean(rewards[-10:]):.2f}")
print(f"  Total episodes: {len(rewards)}")
print(f"\nModel saved to: best_cheetah.pth")
print(f"Training plot saved to: training_results.png")

In [ ]:
# Cell 7: Create and Display Video of Your Trained Cheetah
!apt-get install -y xvfb ffmpeg  # For headless rendering
!pip install imageio[ffmpeg] pyvirtualdisplay

import imageio
import numpy as np
import gymnasium as gym
from IPython.display import Video, HTML, display
import base64
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont  # ADD THIS LINE - import Image as well

# Setup virtual display for headless rendering (needed for Colab)
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1024, 768))
display.start()

def create_and_show_video(episodes=1, fps=30):
    """
    Create a video of the trained cheetah and display it in Colab
    """
    # Create environment with rendering
    base_env = gym.make(config.env_name, render_mode='rgb_array')
    env = AggressiveShapedHalfCheetah(
        base_env,
        forward_reward_scale=config.forward_reward_scale,
        control_penalty=config.control_penalty,
        survival_bonus=config.survival_bonus
    )

    # Load trained agent
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    max_action = float(env.action_space.high[0])

    agent = SACAgent(state_dim, action_dim, max_action, config, device)
    agent.load("best_cheetah.pth")
    agent.actor.eval()

    all_frames = []
    episode_rewards = []

    for episode in range(episodes):
        print(f"Recording episode {episode + 1}/{episodes}...")
        state, _ = env.reset()
        done = False
        frames = []
        episode_reward = 0
        step = 0

        while not done and step < 1000:
            # Get action from trained agent
            action = agent.select_action(state, evaluate=True)

            # Take step
            state, reward_shaped, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            # Track rewards
            episode_reward += info.get('original_reward', reward_shaped)

            # Capture frame
            frame = env.render()
            if frame is not None:
                # Add text overlay
                frame_pil = Image.fromarray(frame)  # Now Image is defined!
                draw = ImageDraw.Draw(frame_pil)
                draw.text((10, 10), f"Step: {step} | Reward: {episode_reward:.1f}",
                         fill=(255, 255, 255))
                frames.append(np.array(frame_pil))

            step += 1

            # Progress indicator
            if step % 100 == 0:
                print(f"  Step {step}/1000, Reward: {episode_reward:.1f}")

        episode_rewards.append(episode_reward)
        all_frames.extend(frames)
        print(f"✅ Episode {episode + 1} complete! Reward: {episode_reward:.2f}")
        print(f"   Frames captured: {len(frames)}")

    env.close()
    base_env.close()

    # Create video
    video_path = "cheetah_trained.mp4"
    print(f"\nCreating video: {video_path}")
    imageio.mimsave(video_path, all_frames, fps=fps, codec='libx264')

    print(f"✅ Video saved! Size: {len(all_frames)} frames at {fps} FPS")
    print(f"   Duration: {len(all_frames)/fps:.1f} seconds")

    # Display video
    from IPython.display import Video as VideoDisplay
    return VideoDisplay(video_path, width=800, height=400), episode_rewards

# Create and display the video
video, rewards = create_and_show_video(episodes=1, fps=30)

print("\n" + "="*50)
print("🎬 YOUR TRAINED CHEETAH VIDEO")
print("="*50)
print(f"Episode reward: {rewards[0]:.2f}")
print(f"Average forward speed: {rewards[0]/1000:.2f} m/s")
print("\nWatch the video above to see your cheetah run!")
print("\n💡 Tips:")
print("  - The cheetah should run smoothly from left to right")
print("  - Look for a consistent galloping gait")
print("  - The white text shows cumulative reward")

In [ ]:
# Cell 8: Clean Video Display
import IPython.display as ipd
import base64
from pathlib import Path

# Reset the display function reference
display_func = ipd.display

video_path = "cheetah_trained.mp4"
if Path(video_path).exists():
    file_size = Path(video_path).stat().st_size / (1024 * 1024)
    print(f"✅ Video file found: {video_path}")
    print(f"   File size: {file_size:.2f} MB")
    print(f"   Duration: 33.3 seconds")

    # Method 1: Try direct video
    print("\n" + "="*50)
    print("ATTEMPTING VIDEO PLAYBACK")
    print("="*50)

    # Use HTML5 video with base64 (most reliable)
    print(" Loading video...")
    with open(video_path, 'rb') as f:
        video_data = f.read()

    video_b64 = base64.b64encode(video_data).decode()
    video_html = f'''
    <html>
    <body>
    <div style="text-align: center;">
        <h3> Your Trained HalfCheetah in Action! </h3>
        <video width="850" height="450" controls autoplay loop>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
            Your browser does not support the video tag.
        </video>
        <p><b>Running Speed: ~3.4 m/s | Total Reward: 3393</b></p>
        <p> Cheetah runs from left to right | Smooth galloping gait</p>
    </div>
    </body>
    </html>
    '''

    # Display using HTML
    display_func(ipd.HTML(video_html))

    # Also provide download link
    from IPython.display import FileLink
    print(f"\n Download video file: {FileLink(video_path)}")

else:
    print("❌ Video file not found. Let's create a GIF instead...")

    import imageio
    import gymnasium as gym

    print("Recreating as GIF (more compatible)...")

    base_env = gym.make(config.env_name, render_mode='rgb_array')
    env = AggressiveShapedHalfCheetah(base_env, **{
        'forward_reward_scale': config.forward_reward_scale,
        'control_penalty': config.control_penalty,
        'survival_bonus': config.survival_bonus
    })

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    max_action = float(env.action_space.high[0])
    agent = SACAgent(state_dim, action_dim, max_action, config, device)
    agent.load("best_cheetah.pth")

    frames = []
    state, _ = env.reset()
    done = False
    step = 0

    while len(frames) < 200 and not done and step < 500:
        action = agent.select_action(state, evaluate=True)
        state, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        if step % 2 == 0:  # Take every 2nd frame
            frame = env.render()
            if frame is not None:
                frames.append(frame)
        step += 1

    env.close()
    base_env.close()

    gif_path = "cheetah_running.gif"
    imageio.mimsave(gif_path, frames, fps=25)
    print(f"✅ GIF created: {gif_path}")

    display_func(ipd.Image(filename=gif_path, width=800))
    print(f"\n Download GIF: {FileLink(gif_path)}")

print("\n" + "="*50)
print("✅ Display complete. You should see the video above.")